In [ ]:
import asyncio, logging, random, re, time, json
from html import unescape

import nest_asyncio
nest_asyncio.apply()

from curl_cffi.requests import AsyncSession
from selectolax.parser import HTMLParser

# ── logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)
logging.getLogger("hpack").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("curl_cffi").setLevel(logging.WARNING)

# ── proxy ─────────────────────────────────────────────────────────────────────
PROXY_HOST = "brd.superproxy.io"
PROXY_PORT  = 33335
PROXY_USER  = "brd-customer-hl_14d32bce-zone-datacenter_proxy1"
PROXY_PASS  = "ymg5cg3a1z33"

def make_proxies():
    token = f"{random.random():.16f}"
    user  = f"{PROXY_USER}-session-{token}"
    url   = f"http://{user}:{PROXY_PASS}@{PROXY_HOST}:{PROXY_PORT}"
    return user, {"http": url, "https": url}

BASE_URL   = "https://www.it-resell.com"
COLLECTION = "/en/collections/all"
SORT       = "price-ascending"

HEADERS = {
    "accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "accept-language": "en-US,en;q=0.9",
    "referer":         "https://www.google.com/",
}

SAMPLE_PAGES = [1, 2, 3]

# ── fetch ─────────────────────────────────────────────────────────────────────
async def fetch(session: AsyncSession, url: str):
    proxy_user, proxies = make_proxies()
    log.debug("GET %s  proxy_user=%s", url, proxy_user)
    t0 = time.perf_counter()
    try:
        r = await session.get(
            url,
            headers=HEADERS,
            proxies=proxies,
            timeout=30,
        )
        elapsed = time.perf_counter() - t0
        log.debug("← %d  size=%d bytes  time=%.2fs  url=%s",
                  r.status_code, len(r.content), elapsed, url)
        if r.status_code != 200:
            log.warning("non-200 %d — preview:\n%s", r.status_code, r.text[:400])
        r.raise_for_status()
        return r.text, len(r.content), elapsed
    except Exception as e:
        log.error("fetch FAILED url=%s  %s: %s", url, type(e).__name__, e)
        raise

# ── parsers ───────────────────────────────────────────────────────────────────
def get_last_page(html: str) -> int:
    tree  = HTMLParser(html)
    links = tree.css("li.pagination_el a[href]")
    last  = 1
    for a in links:
        m = re.search(r'page=(\d+)', a.attributes.get("href", ""))
        if m:
            last = max(last, int(m.group(1)))
    log.debug("get_last_page: %d", last)
    return last

def get_total_products(html: str) -> int:
    tree = HTMLParser(html)
    p    = tree.css_first("div.pagination p")
    if not p:
        log.debug("get_total_products: not found")
        return 0
    text = p.text(strip=True)
    log.debug("get_total_products: raw=%r", text)
    m = re.search(r'of\s+([\d,]+)', text)
    val = int(m.group(1).replace(",", "")) if m else 0
    log.debug("get_total_products: %d", val)
    return val

def parse_price(raw: str | None):
    if not raw:
        return None
    s = unescape(raw)
    s = re.sub(r'<[^>]+>', ' ', s)  # strip any embedded tags
    s = s.replace('\xa0', ' ').replace('€', '')
    s = s.replace('ab', '').strip()

    # remove thousands separators in EU style: 1.299,00 → 1299,00
    s = re.sub(r'(?<=\d)[.](?=\d{3}\b)', '', s)
    s = s.replace(',', '.')

    m = re.search(r'\d+(?:\.\d+)?', s)
    if not m:
        return None
    return float(m.group())

def parse_availability(card) -> str:
    if card.css_first("a.btn_options"):
        return "multi_variant"
    btn = card.css_first("button.btn-cart")
    if btn and btn.attributes.get("disabled") is not None:
        return "unavailable"
    if btn:
        return "in_stock"
    return "unknown"

def parse_products(html: str) -> list[dict]:
    tree     = HTMLParser(html)
    cards    = tree.css("div.product_item")
    products = []

    for card in cards:
        handle   = card.attributes.get("id", "").replace("product__", "").strip()
        url_node = card.css_first("a.img_change")
        path     = url_node.attributes.get("href", "") if url_node else ""
        full_url = f"{BASE_URL}{path}" if path else None

        vendor    = card.css_first("p.product_vendor")
        brand     = vendor.text(strip=True) if vendor else None

        name_node = card.css_first("p.product_name_list a")
        name      = name_node.text(strip=True) if name_node else None

        desc_node = card.css_first("p.product_desc")
        desc      = desc_node.text(strip=True) if desc_node else None

        mpn = ean = manufacturer = None
        for sku_div in card.css("div.single_product__sku"):
            label = sku_div.css_first("b")
            span  = sku_div.css_first("span")
            if not label or not span:
                continue
            key = label.text(strip=True).rstrip(":")
            val = span.text(strip=True)
            if key == "Hersteller Nummer":
                mpn = val
            elif key == "EAN":
                ean = val
            elif key == "Hersteller":
                manufacturer = val

        price_spans = card.css("span.money.main_price")
        prices = []
        for span in price_spans:
            raw = span.attributes.get("data-currency-eur") or span.text()
            p = parse_price(raw)
            if p is not None:
                prices.append(p)

        price_min = prices[0] if len(prices) >= 1 else None
        price_max = prices[1] if len(prices) >= 2 else price_min

        availability = parse_availability(card)
        vid_input    = card.css_first("input[name='id']")
        variant_id   = vid_input.attributes.get("value") if vid_input else None

        products.append({
            "handle":       handle,
            "product_url":  full_url,
            "brand":        brand,
            "manufacturer": manufacturer,
            "name":         name,
            "description":  desc,
            "mpn":          mpn,
            "ean":          ean,
            "price_min":    price_min,
            "price_max":    price_max,
            "availability": availability,
            "variant_id":   variant_id,
            "pdp_required": availability == "multi_variant",
        })

    log.debug("parse_products: found %d cards", len(products))
    return products

# ── discovery ─────────────────────────────────────────────────────────────────
async def discovery():

    total_bytes  = 0
    page_times   = []
    sample_sizes = []
    all_products = []

    async with AsyncSession(impersonate="chrome124") as session:

        # ── step 1: proxy health check ────────────────────────────────────────
        log.info("=== STEP 1: proxy health check ===")
        try:
            _, nb, t = await fetch(session, "https://geo.brdtest.com/welcome.txt")
            log.info("proxy health check PASSED  size=%d  time=%.2fs", nb, t)
        except Exception:
            log.critical("proxy health check FAILED — aborting discovery")
            return

        # ── step 2: fetch page 1 ──────────────────────────────────────────────
        log.info("=== STEP 2: page 1 probe ===")
        url = f"{BASE_URL}{COLLECTION}?sort_by={SORT}&page=1"
        html, nb, t = await fetch(session, url)
        total_bytes  += nb
        page_times.append(t)
        sample_sizes.append(nb)

        total_products = get_total_products(html)
        last_page      = get_last_page(html)
        products_p1    = parse_products(html)
        all_products.extend(products_p1)

        log.info("total_products=%d  last_page=%d  parsed=%d",
                 total_products, last_page, len(products_p1))

        # ── step 3: sample additional pages ───────────────────────────────────
        log.info("=== STEP 3: sampling pages %s ===", SAMPLE_PAGES[1:])
        for page in SAMPLE_PAGES[1:]:
            await asyncio.sleep(random.uniform(1.0, 2.0))
            purl = f"{BASE_URL}{COLLECTION}?sort_by={SORT}&page={page}"
            try:
                html, nb, t = await fetch(session, purl)
                total_bytes  += nb
                page_times.append(t)
                sample_sizes.append(nb)
                prods = parse_products(html)
                all_products.extend(prods)
                log.info("page=%d  parsed=%d  size=%d bytes  time=%.2fs",
                         page, len(prods), nb, t)
            except Exception:
                log.warning("page %d failed — skipping from sample", page)

    # ── step 4: report ────────────────────────────────────────────────────────
    log.info("=== STEP 4: building report ===")

    avg_size  = (sum(sample_sizes) / len(sample_sizes)) if sample_sizes else 0
    avg_time  = (sum(page_times)   / len(page_times))   if page_times   else 0
    est_bytes = avg_size * last_page
    est_gb    = est_bytes / 1024**3 if est_bytes else 0

    pdp_needed    = [p for p in all_products if p["pdp_required"]]
    unavailable   = [p for p in all_products if p["availability"] == "unavailable"]
    in_stock      = [p for p in all_products if p["availability"] == "in_stock"]
    null_price    = [p for p in all_products if p["price_min"] is None]
    ranged_price  = [p for p in all_products
                     if p["price_min"] != p["price_max"] and p["price_min"] is not None]
    single_price  = [p for p in all_products
                     if p["price_min"] == p["price_max"] and p["price_min"] is not None]

    n = len(all_products)
    pct = lambda x: f"{len(x)/n*100:.0f}%" if n else "n/a"

    est_runtime_min = (last_page / 4) * avg_time / 60 if avg_time else 0

    print(f"""
╔══════════════════════════════════════════════════════════════════╗
  it-resell.com — DISCOVERY REPORT
╚══════════════════════════════════════════════════════════════════╝

CATALOGUE
  Total products            : {total_products:,}
  Total pages               : {last_page:,}
  Products per page         : 6
  Sort used                 : {SORT}

PROXY
  Stack                     : curl_cffi + Bright Data residential
  Auth method               : proxy URL user:pass (per-request rotation)
  Health check              : PASSED

SAMPLE  ({len(sample_sizes)} pages)
  Avg page size             : {avg_size/1024:.1f} KB
  Avg response time         : {avg_time:.2f}s
  Products parsed           : {n}

REQUEST BUDGET  (full run)
  Total PLP requests        : {last_page:,}
  Est. total bandwidth      : {est_gb:.2f} GB  ({est_bytes/1024**2:.0f} MB)
  Discovery consumed        : {total_bytes/1024:.1f} KB

AVAILABILITY BREAKDOWN  (sample, n={n})
  In stock                  : {len(in_stock)}  ({pct(in_stock)})
  Multi-variant             : {len(pdp_needed)}  ({pct(pdp_needed)})
  Unavailable               : {len(unavailable)}  ({pct(unavailable)})

PRICE BREAKDOWN  (sample, n={n})
  Single price              : {len(single_price)}  ({pct(single_price)})
  Price range (min-max)     : {len(ranged_price)}  ({pct(ranged_price)})
  Null / zero               : {len(null_price)}  ({pct(null_price)})

PDP REQUIREMENT
  Multi-variant products    : {len(pdp_needed)}  ({pct(pdp_needed)}) of sample
  Est. across full catalogue: ~{int(len(pdp_needed)/n*total_products):,} products
  → PLP gives price range only. PDP needed for variant-level breakdown.
  → Decision required from client before orchestrator build.

CONCURRENCY PLAN  (4 workers, band split)
  Worker 1                  : pages   1 – {last_page//4}
  Worker 2                  : pages   {last_page//4+1} – {last_page//2}
  Worker 3                  : pages   {last_page//2+1} – {3*last_page//4}
  Worker 4                  : pages   {3*last_page//4+1} – {last_page}
  Est. runtime @ 4 workers  : {est_runtime_min:.1f} min

SAMPLE PRODUCTS
""")

    for p in all_products[:3]:
        print(f"  handle       : {p['handle']}")
        print(f"  name         : {p['name']}")
        print(f"  brand        : {p['brand']}")
        print(f"  mpn          : {p['mpn']}")
        print(f"  ean          : {p['ean']}")
        print(f"  price        : {p['price_min']} – {p['price_max']}")
        print(f"  availability : {p['availability']}")
        print(f"  pdp_required : {p['pdp_required']}")
        print()

    # ── save JSON output ──────────────────────────────────────────────────────
    ts = time.strftime("%Y%m%d_%H%M%S")
    output_path = f"it_resell_sample_{ts}.json"
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(all_products, f, indent=2, ensure_ascii=False)
    log.info("saved %d products to %s", len(all_products), output_path)

asyncio.run(discovery())


04:55:29 [INFO] === STEP 1: proxy health check ===
04:55:29 [DEBUG] GET https://geo.brdtest.com/welcome.txt  proxy_user=brd-customer-hl_14d32bce-zone-datacenter_proxy1-session-0.8634591760602294
04:55:31 [DEBUG] ← 200  size=654 bytes  time=1.83s  url=https://geo.brdtest.com/welcome.txt
04:55:31 [INFO] proxy health check PASSED  size=654  time=1.83s
04:55:31 [INFO] === STEP 2: page 1 probe ===
04:55:31 [DEBUG] GET https://www.it-resell.com/en/collections/all?sort_by=price-ascending&page=1  proxy_user=brd-customer-hl_14d32bce-zone-datacenter_proxy1-session-0.8573117423122819
04:55:34 [DEBUG] ← 200  size=384341 bytes  time=2.35s  url=https://www.it-resell.com/en/collections/all?sort_by=price-ascending&page=1
04:55:34 [DEBUG] get_total_products: raw='1 – 6 product(s) of 4435'
04:55:34 [DEBUG] get_total_products: 4435
04:55:34 [DEBUG] get_last_page: 740
04:55:34 [DEBUG] parse_products: found 6 cards
04:55:34 [INFO] total_products=4435  last_page=740  parsed=6
04:55:34 [INFO] === STEP 3: sam


╔══════════════════════════════════════════════════════════════════╗
  it-resell.com — DISCOVERY REPORT
╚══════════════════════════════════════════════════════════════════╝

CATALOGUE
  Total products            : 4,435
  Total pages               : 740
  Products per page         : 6
  Sort used                 : price-ascending

PROXY
  Stack                     : curl_cffi + Bright Data residential
  Auth method               : proxy URL user:pass (per-request rotation)
  Health check              : PASSED

SAMPLE  (3 pages)
  Avg page size             : 379.8 KB
  Avg response time         : 3.08s
  Products parsed           : 18

REQUEST BUDGET  (full run)
  Total PLP requests        : 740
  Est. total bandwidth      : 0.27 GB  (274 MB)
  Discovery consumed        : 1139.4 KB

AVAILABILITY BREAKDOWN  (sample, n=18)
  In stock                  : 9  (50%)
  Multi-variant             : 9  (50%)
  Unavailable               : 0  (0%)

PRICE BREAKDOWN  (sample, n=18)
  Single price    

In [26]:
import asyncio, re, json, time
from html import unescape

from curl_cffi.requests import AsyncSession
from selectolax.parser import HTMLParser

PROXY_HOST = "brd.superproxy.io"
PROXY_PORT  = 33335
PROXY_USER  = "brd-customer-hl_14d32bce-zone-datacenter_proxy1"
PROXY_PASS  = "ymg5cg3a1z33"

BASE_URL = "https://www.it-resell.com"
URL      = f"{BASE_URL}/en/collections/all?sort_by=price-ascending&page=1"

HEADERS = {
    "accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "accept-language": "en-US,en;q=0.9",
    "referer":         "https://www.google.com/",
}

def make_proxy_url():
    import random
    token = f"{random.random():.16f}"
    user  = f"{PROXY_USER}-session-{token}"
    url   = f"http://{user}:{PROXY_PASS}@{PROXY_HOST}:{PROXY_PORT}"
    return {"http": url, "https": url}

def parse_price(raw):
    if not raw:
        return None
    s = unescape(raw)
    s = re.sub(r'<[^>]+>', ' ', s)
    s = s.replace('\xa0', ' ').replace('€', '').replace('ab', '').strip()
    s = re.sub(r'(?<=\d)[.](?=\d{3}\b)', '', s)  # EU thousands: 1.299 → 1299
    s = s.replace(',', '.')
    m = re.search(r'\d+(?:\.\d+)?', s)
    return float(m.group()) if m else None

def parse_availability(card):
    if card.css_first("a.btn_options"):
        return "multi_variant"
    btn = card.css_first("button.btn-cart")
    if btn and btn.attributes.get("disabled") is not None:
        return "unavailable"
    if btn:
        return "in_stock"
    return "unknown"

def parse_products(html, source_url):
    tree     = HTMLParser(html)
    cards    = tree.css("div.product_item")
    products = []

    for card in cards:
        handle    = card.attributes.get("id", "").replace("product__", "").strip()
        url_node  = card.css_first("a.img_change")
        path      = url_node.attributes.get("href", "") if url_node else ""
        full_url  = f"{BASE_URL}{path}" if path else None

        brand     = (card.css_first("p.product_vendor") or type("", (), {"text": lambda *a, **k: None})()).text(strip=True)
        name_node = card.css_first("p.product_name_list a")
        name      = name_node.text(strip=True) if name_node else None
        desc_node = card.css_first("p.product_desc")
        desc      = desc_node.text(strip=True) if desc_node else None

        mpn = ean = manufacturer = None
        for sku_div in card.css("div.single_product__sku"):
            label = sku_div.css_first("b")
            span  = sku_div.css_first("span")
            if not label or not span:
                continue
            key = label.text(strip=True).rstrip(":")
            val = span.text(strip=True)
            if key == "Hersteller Nummer": mpn = val
            elif key == "EAN":             ean = val
            elif key == "Hersteller":      manufacturer = val

        # DOM order: first span = refurbished (lower), second span = new (higher)
        price_spans = card.css("span.money.main_price")
        prices = []
        for span in price_spans:
            raw = span.attributes.get("data-currency-eur") or span.text()
            p = parse_price(raw)
            if p is not None:
                prices.append(p)

        price_refurbished = prices[0] if len(prices) >= 1 else None
        price_new         = prices[1] if len(prices) >= 2 else price_refurbished
        price_zero_flag   = bool(prices and min(p for p in prices if p is not None) == 0.0)

        availability = parse_availability(card)
        vid_input    = card.css_first("input[name='id']")
        variant_id   = vid_input.attributes.get("value") if vid_input else None

        products.append({
            "handle":            handle,
            "product_url":       full_url,
            "brand":             brand,
            "manufacturer":      manufacturer,
            "name":              name,
            "description":       desc,
            "mpn":               mpn,
            "ean":               ean,
            "price_refurbished": price_refurbished,
            "price_new":         price_new,
            "price_zero_flag":   price_zero_flag,
            "availability":      availability,
            "variant_id":        variant_id,
            "source_url":        source_url,
        })

    return products

async def main():
    async with AsyncSession(impersonate="chrome124") as session:
        print(f"Fetching page 1...")
        r = await session.get(URL, headers=HEADERS, proxies=make_proxy_url(), timeout=30)
        r.raise_for_status()
        print(f"Status: {r.status_code}  Size: {len(r.content)} bytes")

        products = parse_products(r.text, source_url=URL)
        print(f"Parsed {len(products)} products\n")

        for p in products:
            print(f"  {p['handle']}")
            print(f"       name              : {p['name']}")
            print(f"       price_refurbished : {p['price_refurbished']}")
            print(f"       price_new         : {p['price_new']}")
            print(f"       price_zero_flag   : {p['price_zero_flag']}")
            print(f"       availability      : {p['availability']}")
            print(f"       source_url        : {p['source_url']}")
            print()

        ts = time.strftime("%Y%m%d_%H%M%S")
        out = f"it_resell_page1_{ts}.json"
        with open(out, "w", encoding="utf-8") as f:
            json.dump(products, f, indent=2, ensure_ascii=False)
        print(f"Saved → {out}")

asyncio.run(main())

Fetching page 1...
Status: 200  Size: 323383 bytes
Parsed 6 products

  hpe-jd368b
       name              : JD368B - HPE 2-port 10GbE SFP+ Module
       price_refurbished : 0.0
       price_new         : 0.0
       price_zero_flag   : True
       availability      : in_stock
       source_url        : https://www.it-resell.com/en/collections/all?sort_by=price-ascending&page=1

  cisco-wic-blank-panel
       name              : WIC-BLANK-PANEL - Cisco Frontabdeckung, schwarz, für Netzwerkgeräte
       price_refurbished : 1.99
       price_new         : 2.99
       price_zero_flag   : False
       availability      : multi_variant
       source_url        : https://www.it-resell.com/en/collections/all?sort_by=price-ascending&page=1

  cisco-ir807-dinrail
       name              : IR807-DINRAIL= - Cisco Industrial DIN Rail Mounting Kit
       price_refurbished : 6.99
       price_new         : 7.99
       price_zero_flag   : False
       availability      : multi_variant
       source_

In [29]:
"""
it-resell.com — Full Catalogue Scraper
======================================
Crawls ALL pages of /en/collections/all?sort_by=price-ascending
using 4 concurrent workers with band-split pagination.

Output: itresell_products.json  (array of product objects)

Proxy: Bright Data datacenter zone
Auth:  embedded in proxy URL (curl_cffi is reliable this way)
"""

import asyncio
import json
import logging
import random
import re
import time
from html import unescape
from pathlib import Path

import nest_asyncio
nest_asyncio.apply()

from curl_cffi.requests import AsyncSession
from selectolax.parser import HTMLParser

# ── logging ──────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)
logging.getLogger("hpack").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("curl_cffi").setLevel(logging.WARNING)

# ── proxy ─────────────────────────────────────────────────────────────────────
PROXY_HOST = "brd.superproxy.io"
PROXY_PORT = 33335
PROXY_USER = "brd-customer-hl_14d32bce-zone-datacenter_proxy1"
PROXY_PASS = "ymg5cg3a1z33"

def make_session_user(token: str = None) -> str:
    t = token or f"{random.random():.16f}"
    return f"{PROXY_USER}-session-{t}"

def make_proxies(session_token: str) -> dict:
    user = make_session_user(session_token)
    url = f"http://{user}:{PROXY_PASS}@{PROXY_HOST}:{PROXY_PORT}"
    return {"http": url, "https": url}

# ── constants ─────────────────────────────────────────────────────────────────
BASE_URL    = "https://www.it-resell.com"
COLLECTION  = "/en/collections/all"
SORT        = "price-ascending"
NUM_WORKERS = 4
OUTPUT_FILE = Path("itresell_products.json")

HEADERS = {
    "accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "accept-language": "en-US,en;q=0.9",
    "referer":         "https://www.google.com/",
}

# ── fetch ─────────────────────────────────────────────────────────────────────
async def fetch(session: AsyncSession, url: str,
                retries: int = 3) -> tuple[str, int, float]:
    for attempt in range(1, retries + 1):
        t0 = time.perf_counter()
        try:
            token = f"{random.random():.16f}"
            proxies = make_proxies(token)
            r = await session.get(
                url,
                headers=HEADERS,
                proxies=proxies,
                timeout=45,
            )
            elapsed = time.perf_counter() - t0
            if r.status_code != 200:
                log.warning("attempt=%d  non-200 %d — %s", attempt, r.status_code, url)
                if attempt < retries:
                    await asyncio.sleep(2 ** attempt)
                    continue
            r.raise_for_status()
            return r.text, len(r.content), elapsed
        except Exception as e:
            log.warning("attempt=%d  fetch FAILED %s — %s: %s", attempt, url, type(e).__name__, e)
            if attempt < retries:
                await asyncio.sleep(2 ** attempt)
            else:
                raise

# ── parsers ───────────────────────────────────────────────────────────────────
def get_last_page(html: str) -> int:
    tree  = HTMLParser(html)
    links = tree.css("li.pagination_el a[href]")
    last  = 1
    for a in links:
        m = re.search(r'page=(\d+)', a.attributes.get("href", ""))
        if m:
            last = max(last, int(m.group(1)))
    return last

def get_total_products(html: str) -> int:
    tree = HTMLParser(html)
    p    = tree.css_first("div.pagination p")
    if not p:
        return 0
    m = re.search(r'of\s+([\d,]+)', p.text(strip=True))
    return int(m.group(1).replace(",", "")) if m else 0

def parse_price(raw):
    if not raw:
        return None
    s = unescape(raw)
    s = re.sub(r'<[^>]+>', ' ', s)
    s = s.replace('\xa0', ' ').replace('€', '').replace('ab', '').strip()
    s = re.sub(r'(?<=\d)[.](?=\d{3}\b)', '', s)  # EU thousands: 1.299 → 1299
    s = s.replace(',', '.')
    m = re.search(r'\d+(?:\.\d+)?', s)
    return float(m.group()) if m else None

def parse_availability(card):
    if card.css_first("a.btn_options"):
        return "multi_variant"
    btn = card.css_first("button.btn-cart")
    if btn and btn.attributes.get("disabled") is not None:
        return "unavailable"
    if btn:
        return "in_stock"
    return "unknown"

def parse_products(html, source_url):
    tree     = HTMLParser(html)
    cards    = tree.css("div.product_item")
    products = []

    for card in cards:
        handle    = card.attributes.get("id", "").replace("product__", "").strip()
        url_node  = card.css_first("a.img_change")
        path      = url_node.attributes.get("href", "") if url_node else ""
        full_url  = f"{BASE_URL}{path}" if path else None

        vendor    = card.css_first("p.product_vendor")
        brand     = vendor.text(strip=True) if vendor else None

        name_node = card.css_first("p.product_name_list a")
        name      = name_node.text(strip=True) if name_node else None

        desc_node = card.css_first("p.product_desc")
        desc      = desc_node.text(strip=True) if desc_node else None

        mpn = ean = manufacturer = None
        for sku_div in card.css("div.single_product__sku"):
            label = sku_div.css_first("b")
            span  = sku_div.css_first("span")
            if not label or not span:
                continue
            key = label.text(strip=True).rstrip(":")
            val = span.text(strip=True)
            if key == "Hersteller Nummer": mpn = val
            elif key == "EAN":             ean = val
            elif key == "Hersteller":      manufacturer = val

        # DOM order: first span = refurbished (lower), second span = new (higher)
        price_spans = card.css("span.money.main_price")
        prices = []
        for span in price_spans:
            raw = span.attributes.get("data-currency-eur") or span.text()
            p = parse_price(raw)
            if p is not None:
                prices.append(p)

        price_refurbished = prices[0] if len(prices) >= 1 else None
        price_new         = prices[1] if len(prices) >= 2 else price_refurbished
        price_zero_flag   = bool(prices and min(p for p in prices if p is not None) == 0.0)

        availability = parse_availability(card)
        vid_input    = card.css_first("input[name='id']")
        variant_id   = vid_input.attributes.get("value") if vid_input else None

        products.append({
            "handle":            handle,
            "product_url":       full_url,
            "brand":             brand,
            "manufacturer":      manufacturer,
            "name":              name,
            "description":       desc,
            "mpn":               mpn,
            "ean":               ean,
            "price_refurbished": price_refurbished,
            "price_new":         price_new,
            "price_zero_flag":   price_zero_flag,
            "availability":      availability,
            "variant_id":        variant_id,
            "source_url":        source_url,
        })

    return products

# ── worker ────────────────────────────────────────────────────────────────────
async def worker(worker_id: int, pages: range, token: str) -> list[dict]:
    results = []
    failed  = []

    async with AsyncSession(impersonate="chrome124") as session:
        for page in pages:
            url = f"{BASE_URL}{COLLECTION}?sort_by={SORT}&page={page}"
            try:
                html, nb, t = await fetch(session, url)
                prods = parse_products(html, source_url=url)
                results.extend(prods)
                log.info("[W%d] page=%d  parsed=%d  size=%dKB  time=%.2fs",
                         worker_id, page, len(prods), nb // 1024, t)
            except Exception as e:
                log.error("[W%d] page=%d FAILED permanently: %s", worker_id, page, e)
                failed.append(page)

            await asyncio.sleep(random.uniform(0.8, 2.0))

    if failed:
        log.warning("[W%d] permanently failed pages: %s", worker_id, failed)

    return results

# ── main ──────────────────────────────────────────────────────────────────────
async def main():
    run_token = f"{int(time.time())}"
    log.info("=" * 60)
    log.info("it-resell.com — FULL CATALOGUE SCRAPE")
    log.info("=" * 60)

    # STEP 1: proxy health check
    log.info("STEP 1: proxy health check")
    async with AsyncSession(impersonate="chrome124") as session:
        try:
            _, nb, t = await fetch(session, "https://geo.brdtest.com/welcome.txt")
            log.info("proxy health check PASSED  size=%d  time=%.2fs", nb, t)
        except Exception as e:
            log.critical("proxy health check FAILED — aborting. %s", e)
            return

    # STEP 2: probe page 1 for catalogue size
    log.info("STEP 2: probing page 1 for catalogue size")
    async with AsyncSession(impersonate="chrome124") as session:
        url = f"{BASE_URL}{COLLECTION}?sort_by={SORT}&page=1"
        html, _, _ = await fetch(session, url)

    total_products = get_total_products(html)
    last_page      = get_last_page(html)
    log.info("total_products=%d  last_page=%d", total_products, last_page)

    if last_page == 0:
        log.critical("Could not determine last page — aborting.")
        return

    # STEP 3: split pages across workers
    log.info("STEP 3: splitting %d pages across %d workers", last_page, NUM_WORKERS)
    bands = []
    pages_per_worker = last_page // NUM_WORKERS
    for i in range(NUM_WORKERS):
        start = i * pages_per_worker + 1
        end   = (i + 1) * pages_per_worker if i < NUM_WORKERS - 1 else last_page
        bands.append(range(start, end + 1))
        log.info("  Worker %d: pages %d – %d  (%d pages)", i + 1, start, end, end - start + 1)

    # STEP 4: run workers concurrently
    log.info("STEP 4: starting %d concurrent workers", NUM_WORKERS)
    t_start = time.perf_counter()

    tasks = [worker(i + 1, bands[i], run_token) for i in range(NUM_WORKERS)]
    worker_results = await asyncio.gather(*tasks)

    elapsed = time.perf_counter() - t_start
    log.info("All workers finished in %.1f minutes", elapsed / 60)

    # STEP 5: merge + dedup
    log.info("STEP 5: merging results")
    all_products = []
    seen_handles = set()
    for batch in worker_results:
        for p in batch:
            if p["handle"] and p["handle"] not in seen_handles:
                seen_handles.add(p["handle"])
                all_products.append(p)
            elif not p["handle"]:
                all_products.append(p)

    log.info("Total unique products: %d  (raw: %d)",
             len(all_products), sum(len(b) for b in worker_results))

    # STEP 6: save JSON
    log.info("STEP 6: saving to %s", OUTPUT_FILE)
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(all_products, f, indent=2, ensure_ascii=False)

    log.info("Saved %d products to %s", len(all_products), OUTPUT_FILE)

    # STEP 7: summary
    multi_variant = [p for p in all_products if p["availability"] == "multi_variant"]
    unavailable   = [p for p in all_products if p["availability"] == "unavailable"]
    in_stock      = [p for p in all_products if p["availability"] == "in_stock"]
    null_price    = [p for p in all_products if p["price_refurbished"] is None]
    zero_price    = [p for p in all_products if p["price_zero_flag"]]
    n = len(all_products)

    print(f"""
╔══════════════════════════════════════════════════════════════╗
  it-resell.com — SCRAPE COMPLETE
╚══════════════════════════════════════════════════════════════╝

  Total products scraped  : {n:,}
  Expected (from site)    : {total_products:,}
  Pages crawled           : {last_page:,}
  Runtime                 : {elapsed/60:.1f} min

  In stock                : {len(in_stock):,}  ({len(in_stock)/n*100:.0f}%)
  Multi-variant           : {len(multi_variant):,}  ({len(multi_variant)/n*100:.0f}%)
  Unavailable             : {len(unavailable):,}  ({len(unavailable)/n*100:.0f}%)
  Null price              : {len(null_price):,}  ({len(null_price)/n*100:.0f}%)
  Zero price (flagged)    : {len(zero_price):,}  ({len(zero_price)/n*100:.0f}%)

  Output file             : {OUTPUT_FILE}
""")

asyncio.run(main())

02:05:57 [INFO] ============================================================
02:05:57 [INFO] it-resell.com — FULL CATALOGUE SCRAPE
02:05:57 [INFO] ============================================================
02:05:57 [INFO] STEP 1: proxy health check
02:05:59 [INFO] proxy health check PASSED  size=625  time=1.84s
02:05:59 [INFO] STEP 2: probing page 1 for catalogue size
02:06:00 [INFO] total_products=4435  last_page=740
02:06:00 [INFO] STEP 3: splitting 740 pages across 4 workers
02:06:00 [INFO]   Worker 1: pages 1 – 185  (185 pages)
02:06:00 [INFO]   Worker 2: pages 186 – 370  (185 pages)
02:06:00 [INFO]   Worker 3: pages 371 – 555  (185 pages)
02:06:00 [INFO]   Worker 4: pages 556 – 740  (185 pages)
02:06:00 [INFO] STEP 4: starting 4 concurrent workers
02:06:01 [INFO] [W1] page=1  parsed=6  size=315KB  time=0.69s
02:06:01 [INFO] [W2] page=186  parsed=6  size=356KB  time=0.81s
02:06:01 [INFO] [W3] page=371  parsed=6  size=321KB  time=1.17s
02:06:03 [INFO] [W4] page=556  parsed=6  size


╔══════════════════════════════════════════════════════════════╗
  it-resell.com — SCRAPE COMPLETE
╚══════════════════════════════════════════════════════════════╝

  Total products scraped  : 4,435
  Expected (from site)    : 4,435
  Pages crawled           : 740
  Runtime                 : 9.0 min

  In stock                : 1,705  (38%)
  Multi-variant           : 2,730  (62%)
  Unavailable             : 0  (0%)
  Null price              : 0  (0%)
  Zero price (flagged)    : 1  (0%)

  Output file             : itresell_products.json

